In [1]:
import gc
import os

import numpy as np
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset

from benchmarks import LightningModule, MultiTargetScaler, PrintEpochMetricsCallback
from dataset.pubchemqc import PubChemQCDataset
from gcnn.base_model import GroupEquivariantCNN

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
target_properties = [
    "pubchemqc_homo",
    "pubchemqc_lumo",
    "pubchemqc_hl_gap",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
]
num_tasks = len(target_properties)

model_path_template = "../models/gcnn_1.1M-pac_grid-[PubChemQC]-fold_{fold}"
model_logs_dir = "../logs"
model_logs_template = "gcnn_1.1M-pac_grid-[PubChemQC]-fold_{fold}"

dataset = PubChemQCDataset(
    properties=target_properties, require_quality_flags=False, num_workers=8
)

targets_array = np.array(dataset.targets, dtype=np.float32)

categories = np.array(
    [f"{pac.category}_{pac.count_total_rings}" for pac in dataset.pacs]
)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_results = []

W1006 21:45:31.628000 25524 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Using device: cuda


In [2]:
for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), categories), 1
):
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=categories[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    scaler = MultiTargetScaler()
    train_targets_np = targets_array[train_idx]
    scaler.fit(train_targets_np)

    all_targets_scaled = scaler.transform(targets_array)
    dataset.targets = torch.tensor(all_targets_scaled, dtype=torch.float32)

    train_loader = DataLoader(train_subset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=256, shuffle=False)

    model = GroupEquivariantCNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=32
    )
    lightning_model = LightningModule(model, target_properties, scaler, lr=1e-3)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=100,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[checkpoint_callback, PrintEpochMetricsCallback(target_properties)],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_model_path = checkpoint_callback.best_model_path
    best_lightning_model = LightningModule.load_from_checkpoint(
        best_model_path, model=model, property_names=target_properties, scaler=scaler
    )
    torch.save(
        best_lightning_model.model.state_dict(),
        os.path.join(save_path_dir, "best.ckpt"),
    )

    dataset.targets = targets_array

    lightning_model.cpu()
    best_lightning_model.cpu()

    del train_subset, val_subset, train_loader, val_loader
    del model, lightning_model, best_lightning_model
    del trainer, checkpoint_callback, logger, scaler
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")

D:\Coding\pac-gcnn\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
Using bfloat16 Automatic Mixed Precision (AMP)



======[ Fold 1/5 ]======


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\mo

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ GroupEquivariantCNN │  1.1 M │ train │     0 │
│ 1 │ criterion  │ MSELoss             │      0 │ train │     0 │
│ 2 │ train_rmsd │ MetricDict          │      0 │ train │     0 │
│ 3 │ val_rmsd   │ MetricDict          │      0 │ train │     0 │
│ 4 │ test_rmsd  │ MetricDict          │      0 │ train │     0 │
│ 5 │ test_mae   │ MetricDict          │      0 │ train │     0 │
│ 6 │ test_r2    │ MetricDict          │      0 │ train │     0 │
│ 7 │ test_max   │ MetricDict          │      0 │ train │     0 │
│ 8 │ test_min   │ MetricDict          │      0 │ train │     0 │
└───┴────────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.360                                                                      
Modules in train mode: 53                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.360                                                                      
Modules in train mode: 53                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


Epoch 001 | Scaled Train Loss: N/A | Scaled Val Loss: 6.80405 | pubchemqc_homo Val RMSD: 1.03114 | pubchemqc_lumo Val RMSD: 2.31878 | pubchemqc_hl_gap Val RMSD: 2.59408
Epoch 002 | Scaled Train Loss: 0.45631 | Scaled Val Loss: 6.21006 | pubchemqc_homo Val RMSD: 0.82989 | pubchemqc_lumo Val RMSD: 2.39488 | pubchemqc_hl_gap Val RMSD: 2.33075
Epoch 003 | Scaled Train Loss: 0.19519 | Scaled Val Loss: 0.29431 | pubchemqc_homo Val RMSD: 0.25045 | pubchemqc_lumo Val RMSD: 0.51725 | pubchemqc_hl_gap Val RMSD: 0.48478
Epoch 004 | Scaled Train Loss: 0.12827 | Scaled Val Loss: 0.11039 | pubchemqc_homo Val RMSD: 0.20551 | pubchemqc_lumo Val RMSD: 0.26194 | pubchemqc_hl_gap Val RMSD: 0.32336
Epoch 005 | Scaled Train Loss: 0.09330 | Scaled Val Loss: 0.09353 | pubchemqc_homo Val RMSD: 0.20191 | pubchemqc_lumo Val RMSD: 0.23576 | pubchemqc_hl_gap Val RMSD: 0.29480
Epoch 006 | Scaled Train Loss: 0.07433 | Scaled Val Loss: 0.12906 | pubchemqc_homo Val RMSD: 0.27167 | pubchemqc_lumo Val RMSD: 0.20290 | p

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100 | Scaled Train Loss: 0.00562 | Scaled Val Loss: 0.03255 | pubchemqc_homo Val RMSD: 0.11122 | pubchemqc_lumo Val RMSD: 0.12003 | pubchemqc_hl_gap Val RMSD: 0.19642
Best scaled validation loss for fold 5: 0.0321

======[ Final k-fold results ]======
Scaled Loss per fold: [0.0245, 0.0265, 0.0289, 0.0317, 0.0321]
Average Scaled Loss: 0.0287 ± 0.0030


In [3]:
import gc
import os

import numpy as np
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset

from benchmarks import LightningModule, MultiTargetScaler, PrintEpochMetricsCallback
from dataset.pubchemqc import PubChemQCDataset
from gcnn.base_model import GroupEquivariantCNN

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
target_properties = [
    "b3lyppm6_homo",
    "b3lyppm6_lumo",
    "b3lyppm6_hl_gap",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
]
num_tasks = len(target_properties)

model_path_template = "../models/gcnn_1.1M-pac_grid-[PubChemQC-corrected]-fold_{fold}"
model_logs_dir = "../logs"
model_logs_template = "gcnn_1.1M-pac_grid-[PubChemQC-corrected]-fold_{fold}"

dataset = PubChemQCDataset(
    properties=target_properties, require_quality_flags=False, num_workers=8
)

targets_array = np.array(dataset.targets, dtype=np.float32)

categories = np.array(
    [f"{pac.category}_{pac.count_total_rings}" for pac in dataset.pacs]
)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_results = []

Using device: cuda


In [4]:
for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), categories), 1
):
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=categories[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    scaler = MultiTargetScaler()
    train_targets_np = targets_array[train_idx]
    scaler.fit(train_targets_np)

    all_targets_scaled = scaler.transform(targets_array)
    dataset.targets = torch.tensor(all_targets_scaled, dtype=torch.float32)

    train_loader = DataLoader(train_subset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=256, shuffle=False)

    model = GroupEquivariantCNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=32
    )
    lightning_model = LightningModule(model, target_properties, scaler, lr=1e-3)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=100,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[checkpoint_callback, PrintEpochMetricsCallback(target_properties)],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_model_path = checkpoint_callback.best_model_path
    best_lightning_model = LightningModule.load_from_checkpoint(
        best_model_path, model=model, property_names=target_properties, scaler=scaler
    )
    torch.save(
        best_lightning_model.model.state_dict(),
        os.path.join(save_path_dir, "best.ckpt"),
    )

    dataset.targets = targets_array

    lightning_model.cpu()
    best_lightning_model.cpu()

    del train_subset, val_subset, train_loader, val_loader
    del model, lightning_model, best_lightning_model
    del trainer, checkpoint_callback, logger, scaler
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")

D:\Coding\pac-gcnn\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.



======[ Fold 1/5 ]======


D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\model_summary\model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ GroupEquivariantCNN │  1.1 M │ train │     0 │
│ 1 │ criterion  │ MSELoss             │      0 │ train │     0 │
│ 2 │ train_rmsd │ MetricDict          │      0 │ train │     0 │
│ 3 │ val_rmsd   │ MetricDict          │      0 │ train │     0 │
│ 4 │ test_rmsd  │ MetricDict          │      0 │ train │     0 │
│ 5 │ test_mae   │ MetricDict          │      0 │ train │     0 │
│ 6 │ test_r2    │ MetricDict          │      0 │ train │     0 │
│ 7 │ test_max   │ MetricDict          │      0 │ train │     0 │
│ 8 │ test_min   │ MetricDict          │      0 │ train │     0 │
└───┴────────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.360                                                                      
Modules in train mode: 53                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.360                                                                      
Modules in train mode: 53                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


Epoch 001 | Scaled Train Loss: N/A | Scaled Val Loss: 1.02694 | b3lyppm6_homo Val RMSD: 0.88092 | b3lyppm6_lumo Val RMSD: 0.78607 | b3lyppm6_hl_gap Val RMSD: 0.78375
Epoch 002 | Scaled Train Loss: 0.51750 | Scaled Val Loss: 0.84865 | b3lyppm6_homo Val RMSD: 0.74876 | b3lyppm6_lumo Val RMSD: 0.66077 | b3lyppm6_hl_gap Val RMSD: 0.81500
Epoch 003 | Scaled Train Loss: 0.17733 | Scaled Val Loss: 0.10467 | b3lyppm6_homo Val RMSD: 0.22133 | b3lyppm6_lumo Val RMSD: 0.25517 | b3lyppm6_hl_gap Val RMSD: 0.29471
Epoch 004 | Scaled Train Loss: 0.10821 | Scaled Val Loss: 0.06378 | b3lyppm6_homo Val RMSD: 0.18295 | b3lyppm6_lumo Val RMSD: 0.21577 | b3lyppm6_hl_gap Val RMSD: 0.20610
Epoch 005 | Scaled Train Loss: 0.08006 | Scaled Val Loss: 0.15771 | b3lyppm6_homo Val RMSD: 0.26486 | b3lyppm6_lumo Val RMSD: 0.16922 | b3lyppm6_hl_gap Val RMSD: 0.45636
Epoch 006 | Scaled Train Loss: 0.06251 | Scaled Val Loss: 0.09439 | b3lyppm6_homo Val RMSD: 0.14776 | b3lyppm6_lumo Val RMSD: 0.30613 | b3lyppm6_hl_gap Va

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100 | Scaled Train Loss: 0.00676 | Scaled Val Loss: 0.01411 | b3lyppm6_homo Val RMSD: 0.06601 | b3lyppm6_lumo Val RMSD: 0.09477 | b3lyppm6_hl_gap Val RMSD: 0.11747
Best scaled validation loss for fold 5: 0.0141

======[ Final k-fold results ]======
Scaled Loss per fold: [0.0114, 0.0078, 0.014, 0.011, 0.0141]
Average Scaled Loss: 0.0117 ± 0.0023


In [5]:
import gc
import os

import numpy as np
import torch
from lightning import pytorch as pl
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset

from benchmarks import LightningModule, MultiTargetScaler
from dataset.pubchemqc import PubChemQCDataset
from gcnn.base_model import GroupEquivariantCNN

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
target_properties = [
    "pubchemqc_homo",
    "pubchemqc_lumo",
    "pubchemqc_hl_gap",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
]
num_tasks = len(target_properties)
model_path_template = "../models/gcnn_1.1M-pac_grid-[PubChemQC]-fold_{fold}"

dataset = PubChemQCDataset(
    properties=target_properties, require_quality_flags=False, num_workers=18
)
targets_array = np.array(dataset.targets, dtype=np.float32)

categories = np.array(
    [f"{pac.category}_{pac.count_total_rings}" for pac in dataset.pacs]
)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), categories), 1
):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=categories[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    train_subset = Subset(dataset, train_idx)
    test_subset = Subset(dataset, test_idx)

    scaler = MultiTargetScaler()
    scaler.fit(targets_array[train_idx])

    dataset.targets = torch.tensor(scaler.transform(targets_array), dtype=torch.float32)

    train_loader = DataLoader(train_subset, batch_size=512, shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=512, shuffle=False)

    model = GroupEquivariantCNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=32
    )
    load_path = os.path.join(model_path_template.format(fold=fold), "best.ckpt")
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = LightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("======[ Train Set Evaluation ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("======[ Test Set Evaluation ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for prop in target_properties:
        # Metric names from trainer.test() start with test_
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]

        print(f"======[ {prop} ]======")
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        all_results["train"][prop]["mae"].append(t_mae)
        all_results["train"][prop]["rmsd"].append(t_rmsd)
        all_results["train"][prop]["max"].append(t_max)
        all_results["train"][prop]["min"].append(t_min)
        all_results["train"][prop]["r2"].append(t_r2)

        all_results["test"][prop]["mae"].append(te_mae)
        all_results["test"][prop]["rmsd"].append(te_rmsd)
        all_results["test"][prop]["max"].append(te_max)
        all_results["test"][prop]["min"].append(te_min)
        all_results["test"][prop]["r2"].append(te_r2)

    dataset.targets = targets_array

    del model, lightning_model, trainer, scaler
    del train_results, test_results, train_loader, test_loader
    del train_subset, test_subset
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results (mean ± std) ]======")

for prop in target_properties:
    print(f"\n======[ {prop} ]======")
    for dataset_name, key in [("Train", "train"), ("Test", "test")]:
        m = all_results[key][prop]
        print(f"{dataset_name}:")
        print(f" + MAE: {np.mean(m['mae']):.4f} ± {np.std(m['mae'], ddof=1):.4f}")
        print(f" + RMSD: {np.mean(m['rmsd']):.4f} ± {np.std(m['rmsd'], ddof=1):.4f}")
        print(f" + Max: {np.mean(m['max']):.4f} ± {np.std(m['max'], ddof=1):.4f}")
        print(f" + Min: {np.mean(m['min']):.4f} ± {np.std(m['min'], ddof=1):.4f}")
        print(f" + R^2: {np.mean(m['r2']):.4f} ± {np.std(m['r2'], ddof=1):.4f}")

Using device: cuda


D:\Coding\pac-gcnn\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lig


======[ Evaluating fold 1/5 ]======
======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


======[ Test Set Evaluation ]======
======[ pubchemqc_homo ]======
 + Train -> MAE: 0.0319	RMSD: 0.0421	Max: 0.3043	Min: 0.0000	R^2: 0.9976
 + Test -> MAE: 0.0576	RMSD: 0.0963	Max: 0.9722	Min: 0.0000	R^2: 0.9866
======[ pubchemqc_lumo ]======
 + Train -> MAE: 0.0323	RMSD: 0.0452	Max: 0.5814	Min: 0.0000	R^2: 0.9966
 + Test -> MAE: 0.0635	RMSD: 0.1277	Max: 2.1445	Min: 0.0000	R^2: 0.9711
======[ pubchemqc_hl_gap ]======
 + Train -> MAE: 0.0323	RMSD: 0.0468	Max: 0.4685	Min: 0.0000	R^2: 0.9967
 + Test -> MAE: 0.0839	RMSD: 0.1738	Max: 2.1645	Min: 0.0000	R^2: 0.9557


...

======[ Test Set Evaluation ]======
======[ pubchemqc_homo ]======
 + Train -> MAE: 0.0158	RMSD: 0.0212	Max: 0.1764	Min: 0.0000	R^2: 0.9994
 + Test -> MAE: 0.0523	RMSD: 0.1009	Max: 1.0655	Min: 0.0000	R^2: 0.9857
======[ pubchemqc_lumo ]======
 + Train -> MAE: 0.0177	RMSD: 0.0225	Max: 0.2540	Min: 0.0000	R^2: 0.9991
 + Test -> MAE: 0.0580	RMSD: 0.1232	Max: 1.7338	Min: 0.0000	R^2: 0.9749
======[ pubchemqc_hl_gap ]======
 + Train -> MAE: 0.0181	RMSD: 0.0244	Max: 0.1839	Min: 0.0000	R^2: 0.9991
 + Test -> MAE: 0.0822	RMSD: 0.1820	Max: 2.7223	Min: 0.0000	R^2: 0.9503


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Evaluating fold 5/5 ]======
======[ Train Set Evaluation ]======


D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


======[ Test Set Evaluation ]======
======[ pubchemqc_homo ]======
 + Train -> MAE: 0.0114	RMSD: 0.0149	Max: 0.1052	Min: 0.0000	R^2: 0.9997
 + Test -> MAE: 0.0524	RMSD: 0.1027	Max: 1.1546	Min: 0.0000	R^2: 0.9855
======[ pubchemqc_lumo ]======
 + Train -> MAE: 0.0160	RMSD: 0.0196	Max: 0.1123	Min: 0.0000	R^2: 0.9994
 + Test -> MAE: 0.0568	RMSD: 0.1255	Max: 1.8906	Min: 0.0000	R^2: 0.9735
======[ pubchemqc_hl_gap ]======
 + Train -> MAE: 0.0140	RMSD: 0.0180	Max: 0.1225	Min: 0.0000	R^2: 0.9995
 + Test -> MAE: 0.0836	RMSD: 0.1892	Max: 2.4470	Min: 0.0000	R^2: 0.9467

======[ Final k-fold results (mean ± std) ]======

======[ pubchemqc_homo ]======
Train:
 + MAE: 0.0183 ± 0.0099
 + RMSD: 0.0239 ± 0.0128
 + Max: 0.1684 ± 0.0824
 + Min: 0.0000 ± 0.0000
 + R^2: 0.9990 ± 0.0009
Test:
 + MAE: 0.0546 ± 0.0039
 + RMSD: 0.1035 ± 0.0066
 + Max: 1.1556 ± 0.2368
 + Min: 0.0000 ± 0.0000
 + R^2: 0.9852 ± 0.0016

======[ pubchemqc_lumo ]======
Train:
 + MAE: 0.0184 ± 0.0089
 + RMSD: 0.0244 ± 0.0129
 + Max: 

In [6]:
import gc
import os

import numpy as np
import torch
from lightning import pytorch as pl
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Subset

from benchmarks import LightningModule, MultiTargetScaler
from dataset.pubchemqc import PubChemQCDataset
from gcnn.base_model import GroupEquivariantCNN

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
target_properties = [
    "b3lyppm6_homo",
    "b3lyppm6_lumo",
    "b3lyppm6_hl_gap",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
]
num_tasks = len(target_properties)
model_path_template = "../models/gcnn_1.1M-pac_grid-[PubChemQC-corrected]-fold_{fold}"

dataset = PubChemQCDataset(
    properties=target_properties, require_quality_flags=False, num_workers=18
)
targets_array = np.array(dataset.targets, dtype=np.float32)

categories = np.array(
    [f"{pac.category}_{pac.count_total_rings}" for pac in dataset.pacs]
)

skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(np.zeros(len(dataset)), categories), 1
):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    try:
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=0.125,
            random_state=42,
            stratify=categories[train_val_idx],
        )
    except ValueError:
        train_idx, val_idx = train_test_split(
            train_val_idx, test_size=0.125, random_state=42
        )

    train_subset = Subset(dataset, train_idx)
    test_subset = Subset(dataset, test_idx)

    scaler = MultiTargetScaler()
    scaler.fit(targets_array[train_idx])

    dataset.targets = torch.tensor(scaler.transform(targets_array), dtype=torch.float32)

    train_loader = DataLoader(train_subset, batch_size=512, shuffle=False)
    test_loader = DataLoader(test_subset, batch_size=512, shuffle=False)

    model = GroupEquivariantCNN(
        property_types=property_types, kernel_size=5, num_hidden=3, hidden_channels=32
    )
    load_path = os.path.join(model_path_template.format(fold=fold), "best.ckpt")
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = LightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("======[ Train Set Evaluation ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("======[ Test Set Evaluation ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for prop in target_properties:
        # Metric names from trainer.test() start with test_
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]

        print(f"======[ {prop} ]======")
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        all_results["train"][prop]["mae"].append(t_mae)
        all_results["train"][prop]["rmsd"].append(t_rmsd)
        all_results["train"][prop]["max"].append(t_max)
        all_results["train"][prop]["min"].append(t_min)
        all_results["train"][prop]["r2"].append(t_r2)

        all_results["test"][prop]["mae"].append(te_mae)
        all_results["test"][prop]["rmsd"].append(te_rmsd)
        all_results["test"][prop]["max"].append(te_max)
        all_results["test"][prop]["min"].append(te_min)
        all_results["test"][prop]["r2"].append(te_r2)

    dataset.targets = targets_array

    del model, lightning_model, trainer, scaler
    del train_results, test_results, train_loader, test_loader
    del train_subset, test_subset
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results (mean ± std) ]======")

for prop in target_properties:
    print(f"\n======[ {prop} ]======")
    for dataset_name, key in [("Train", "train"), ("Test", "test")]:
        m = all_results[key][prop]
        print(f"{dataset_name}:")
        print(f" + MAE: {np.mean(m['mae']):.4f} ± {np.std(m['mae'], ddof=1):.4f}")
        print(f" + RMSD: {np.mean(m['rmsd']):.4f} ± {np.std(m['rmsd'], ddof=1):.4f}")
        print(f" + Max: {np.mean(m['max']):.4f} ± {np.std(m['max'], ddof=1):.4f}")
        print(f" + Min: {np.mean(m['min']):.4f} ± {np.std(m['min'], ddof=1):.4f}")
        print(f" + R^2: {np.mean(m['r2']):.4f} ± {np.std(m['r2'], ddof=1):.4f}")

Using device: cuda


D:\Coding\pac-gcnn\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lig


======[ Evaluating fold 1/5 ]======
======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


======[ Test Set Evaluation ]======
======[ b3lyppm6_homo ]======
 + Train -> MAE: 0.0097	RMSD: 0.0127	Max: 0.1199	Min: 0.0000	R^2: 0.9998
 + Test -> MAE: 0.0352	RMSD: 0.0701	Max: 0.8737	Min: 0.0000	R^2: 0.9930
======[ b3lyppm6_lumo ]======
 + Train -> MAE: 0.0095	RMSD: 0.0122	Max: 0.0798	Min: 0.0000	R^2: 0.9998
 + Test -> MAE: 0.0337	RMSD: 0.0829	Max: 2.5203	Min: 0.0000	R^2: 0.9879
======[ b3lyppm6_hl_gap ]======
 + Train -> MAE: 0.0135	RMSD: 0.0161	Max: 0.0803	Min: 0.0000	R^2: 0.9996
 + Test -> MAE: 0.0480	RMSD: 0.1120	Max: 2.5075	Min: 0.0001	R^2: 0.9809


...

======[ Test Set Evaluation ]======
======[ b3lyppm6_homo ]======
 + Train -> MAE: 0.0091	RMSD: 0.0118	Max: 0.0986	Min: 0.0000	R^2: 0.9998
 + Test -> MAE: 0.0375	RMSD: 0.0744	Max: 0.8846	Min: 0.0000	R^2: 0.9922
======[ b3lyppm6_lumo ]======
 + Train -> MAE: 0.0120	RMSD: 0.0162	Max: 0.1241	Min: 0.0000	R^2: 0.9996
 + Test -> MAE: 0.0371	RMSD: 0.0775	Max: 1.4172	Min: 0.0000	R^2: 0.9900
======[ b3lyppm6_hl_gap ]======
 + Train -> MAE: 0.0154	RMSD: 0.0197	Max: 0.1362	Min: 0.0000	R^2: 0.9994
 + Test -> MAE: 0.0496	RMSD: 0.1011	Max: 1.4839	Min: 0.0000	R^2: 0.9839


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Evaluating fold 5/5 ]======
======[ Train Set Evaluation ]======


D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


======[ Test Set Evaluation ]======
======[ b3lyppm6_homo ]======
 + Train -> MAE: 0.0099	RMSD: 0.0133	Max: 0.1090	Min: 0.0000	R^2: 0.9998
 + Test -> MAE: 0.0384	RMSD: 0.0786	Max: 1.2022	Min: 0.0000	R^2: 0.9916
======[ b3lyppm6_lumo ]======
 + Train -> MAE: 0.0089	RMSD: 0.0119	Max: 0.1054	Min: 0.0000	R^2: 0.9998
 + Test -> MAE: 0.0340	RMSD: 0.0666	Max: 1.2085	Min: 0.0000	R^2: 0.9925
======[ b3lyppm6_hl_gap ]======
 + Train -> MAE: 0.0085	RMSD: 0.0113	Max: 0.1122	Min: 0.0000	R^2: 0.9998
 + Test -> MAE: 0.0496	RMSD: 0.1071	Max: 2.3656	Min: 0.0000	R^2: 0.9823

======[ Final k-fold results (mean ± std) ]======

======[ b3lyppm6_homo ]======
Train:
 + MAE: 0.0103 ± 0.0028
 + RMSD: 0.0134 ± 0.0031
 + Max: 0.1076 ± 0.0093
 + Min: 0.0000 ± 0.0000
 + R^2: 0.9997 ± 0.0001
Test:
 + MAE: 0.0369 ± 0.0021
 + RMSD: 0.0743 ± 0.0067
 + Max: 1.0271 ± 0.2508
 + Min: 0.0000 ± 0.0000
 + R^2: 0.9924 ± 0.0013

======[ b3lyppm6_lumo ]======
Train:
 + MAE: 0.0096 ± 0.0014
 + RMSD: 0.0126 ± 0.0020
 + Max: 0.095